In [ ]:
# === ノートブック共通の前処理 (llm_math パッケージの読み込み) ===
import sys
from pathlib import Path

# llm_math パッケージの候補パス
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 親ディレクトリも候補に追加 (notebooks/ フォルダで実行する場合)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math の import を試行
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[注意] llm_math パッケージの読み込みエラー: {e}")
    print("  GitHub リポジトリを clone して colab_setup.sh を実行してください。")
# === 前処理ここまで ===


# Ch 18. GPTの構造

> **学習目標**
> - GPT-2 LLM PyTorch
> - $L$, $h$, 次元 $d$ 計算
> - モデル 順伝播 時間(CPU vs GPU)

## 18.1 GPT

- GPT-1 (2018): 117M params, d=768, L=12
- GPT-2 (2019): 1.5B params, d=1600, L=48
- GPT-3 (2020): 175B params, d=12288, L=96
- GPT-4 (2023): , MoE

: , Pre-LN, GELU FFN, 学習 ,

## 18.2 MiniGPT

 GPT-2 モデル .


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# (ch17  TransformerBlock  —   )
class TransformerBlock(nn.Module):
    """Pre-LN Transformer Block."""
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def attention(self, x, mask):
        B, T, D = x.shape
        q, k, v = self.W_qkv(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        scores = q @ k.transpose(-1, -2) / (self.d_k ** 0.5) + mask
        weights = F.softmax(scores, dim=-1)
        out = (weights @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.W_o(out)

    def forward(self, x, mask):
        x = x + self.dropout(self.attention(self.ln1(x), mask))
        x = x + self.dropout(self.ffn(self.ln2(x)))
        return x

class MiniGPT(nn.Module):
    """GPT-2   Language Model."""
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=4,
                 d_ff=512, max_seq_len=256, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        # weight tying: OutputLayer Embedding Transpose
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T = x.shape
        positions = torch.arange(T, device=x.device).unsqueeze(0)
        # Embedding + Scaling + Position
        emb = self.token_emb(x) * (self.d_model ** 0.5) + self.pos_emb(positions)
        h = self.dropout(emb)
        # Causal Mask
        mask = torch.triu(torch.full((T, T), float('-inf'), device=x.device), diagonal=1)
        for block in self.blocks:
            h = block(h, mask)
        h = self.ln_f(h)
        logits = self.lm_head(h)
        return logits

# Test
vocab_size = 1000
model = MiniGPT(vocab_size, d_model=128, n_heads=4, n_layers=4, d_ff=512)
n_params = sum(p.numel() for p in model.parameters())
print(f"MiniGPT  : {n_params:,}")

x = torch.randint(0, vocab_size, (2, 32))  # batch=2, seq=32
logits = model(x)
print(f"Input: {x.shape}")
print(f"Output : {logits.shape} (batch, seq, vocab)")


## 18.3 Weight Tying

 行列 $\mathbf{E} \in \mathbb{R}^{|V| \times d}$ 出力 $\mathbf{W}_{out} \in \mathbb{R}^{d \times |V|}$ ** 行列** .

:
1. ($|V| \cdot d$ )
2. 出力 空間 →
3. 性能度

GPT-2, LLaMA LLM weight tying .


In [ ]:
# Weight tying 
vocab_size, d = 50000, 768

# tying  
class NoTyingModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d)
        self.head = nn.Linear(d, vocab_size)
    def forward(self, x): return self.head(self.emb(x))

# tying 
class TyingModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d)
        self.head = nn.Linear(d, vocab_size, bias=False)
        self.head.weight = self.emb.weight  # !
    def forward(self, x): return self.head(self.emb(x))

m1 = NoTyingModel()
m2 = TyingModel()
print(f"Tying : {sum(p.numel() for p in m1.parameters()):,} params")
print(f"Tying : {sum(p.numel() for p in m2.parameters()):,} params")
print(f": {(sum(p.numel() for p in m1.parameters()) - sum(p.numel() for p in m2.parameters())):,} params")


## 18.4 [CPU/GPU ベンチマーク ⑦] モデルサイズ別 順伝播/バックプロパゲーション


In [ ]:
# モデルサイズ別 ベンチマーク
import time
from llm_math.bench import time_fn

configs = [
    ('1M',  1000, 64,  2, 4, 256),
    ('5M',  1000, 128, 4, 4, 512),
    ('20M', 1000, 256, 4, 8, 1024),
    ('50M', 1000, 384, 6, 8, 1536),
]

print(f"{'Size':>5} | {'Params':>10} | {'CPU fwd (ms)':>14} | {'CPU fwd+bwd (ms)':>18} | {'GPU fwd (ms)':>14}")
print("-" * 70)
for name, V, d, L, h, d_ff in configs:
    model = MiniGPT(V, d_model=d, n_heads=h, n_layers=L, d_ff=d_ff, max_seq_len=128)
    n_params = sum(p.numel() for p in model.parameters())
    x = torch.randint(0, V, (1, 64))
    target = torch.randint(0, V, (1, 64))
    loss_fn = nn.CrossEntropyLoss()

    def fwd_step():
        with torch.no_grad():
            return model(x)

    def fwd_bwd_step():
        logits = model(x)
        loss = loss_fn(logits.reshape(-1, V), target.reshape(-1))
        loss.backward()
        model.zero_grad()
        return loss

    res_fwd = time_fn(fwd_step, device='cpu', warmup=2, repeat=3)
    res_fb = time_fn(fwd_bwd_step, device='cpu', warmup=1, repeat=2)
    if torch.cuda.is_available():
        model_gpu = model.cuda()
        x_g = x.cuda()
        def fwd_gpu():
            with torch.no_grad():
                return model_gpu(x_g)
        res_gpu = time_fn(fwd_gpu, device='cuda', warmup=2, repeat=3)
        print(f"{name:>5} | {n_params:>10,} | {res_fwd['mean_ms']:>14.3f} | {res_fb['mean_ms']:>18.3f} | {res_gpu['mean_ms']:>14.3f}")
    else:
        print(f"{name:>5} | {n_params:>10,} | {res_fwd['mean_ms']:>14.3f} | {res_fb['mean_ms']:>18.3f} | {'N/A':>14}")


## 18.5

学習 . ( , )


In [ ]:
#    
from llm_math.data import load_tiny_shakespeare

text = load_tiny_shakespeare(max_chars=5000)
chars = sorted(set(text))
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}
vocab_size = len(chars)
print(f"Vocabulary Size: {vocab_size}")

#  GPT
model = MiniGPT(vocab_size, d_model=64, n_heads=4, n_layers=2, d_ff=256, max_seq_len=128)
print(f"Model : {sum(p.numel() for p in model.parameters()):,}")

def generate(model, prompt, n_new=100, temperature=1.0):
    model.eval()
    idx = [char_to_idx[c] for c in prompt]
    with torch.no_grad():
        for _ in range(n_new):
            x = torch.tensor([idx], dtype=torch.long)
            logits = model(x)
            #  
            logit = logits[0, -1] / temperature
            probs = F.softmax(logit, dim=0)
            next_idx = torch.multinomial(probs, 1).item()
            idx.append(next_idx)
    return ''.join(idx_to_char[i] for i in idx)

print("\n===  Weight Generation (Training ) ===")
print(generate(model, "To be", n_new=100, temperature=0.7))
print("\n=> Training   Model   Output.")


## 18.6 要点

| 内容 | MiniGPT | GPT-2 small | LLaMA-7B |
|---|---|---|---|
| d_model | 128 | 768 | 4096 |
| n_layers | 4 | 12 | 32 |
| n_heads | 4 | 12 | 32 |
| d_ff | 512 | 3072 | 11008 |
| | 1000 | 50257 | 32000 |
| | ~1M | ~124M | ~7B |

## 演習問題

1. MiniGPT d_model=64, 128, 256 順伝播 時間 比較.
2. Weight tying 性能 比較.
3. モデル L=2, 4, 8 順伝播 時間 .
4. $\sqrt{d}$ 学習 .
5. (Ch 19) モデル 学習 . 学習 データ .

> 解答: `solutions/ch18_solutions.ipynb`
